
# Generate an EDA Summary Helper
In the previous lesson, you built a generic helper.In this notebook we sharpen that helper for a specific job: **EDA or exploratory data analysis** such as descriptive statistics, missing values, and distributions.

A High Level diagram of the helper
![](../../images/eda_helper_clean_flow.png)



## 1 — Setup and Imports

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [1]:
%pip install -qq google-genai pandas scikit-learn matplotlib seaborn python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
from google import genai
from dotenv import load_dotenv
import pandas as pd

import warnings

warnings.filterwarnings("ignore")

# Load environment variables from a .env file into the environment
load_dotenv()

# Read the Gemini API key from the environment variables
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Create a Gemini client using the API key
# This client will be used to send requests to Gemini models
client = genai.Client(api_key=GEMINI_API_KEY)


### Load the dataset

In [3]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


df = pd.read_csv("../../data/hr_analytics.csv")
df.head()

,Employee ID,age,gender,department,department_code,JobTitle,job_level,Education,MonthlyIncome,monthly_rate,hourly_rate,daily_rate,years_at_company,years_in_role,years_since_promotion,satisfaction_score,environment_satisfaction,Attrition,OverTime,distance_from_home,training_hours_last_year,num_companies_worked,manager_rating,work_life_balance,last_promotion_date
0,1001,27,Female,Engineering,ENG-02,Backend Developer,3,High School,8001,9162,45,369,0,0,0,3 - High,4,No,No,11.0,32.0,1,4.0,Very High,2024-09-06
1,1002,34,Male,Engineering,ENG-02,Data Engineer,2,Master's,8777,10301,51,415,4,3,2,2 - Medium,2,No,No,26.0,30.0,0,2.0,Low,12/18/2022
2,1003,50,Female,Finance,FIN-05,Controller,4,Master's,14021,18470,82,659,13,11,7,4 - Very High,3,No,No,17.0,40.0,2,3.0,Medium,2017-12-04
3,1004,31,Male,Marketing,MKT-04,Marketing Analyst,3,Bachelor's,6402,8805,37,297,3,0,0,4 - Very High,3,No,No,3.0,32.0,0,2.0,High,2024-12-07
4,1005,51,Male,Marketing,MKT-04,SEO Specialist,4,Master's,9277,12125,54,433,21,4,3,4 - Very High,2,No,No,3.0,25.0,0,1.0,High,06/25/2021


## 2 — Context Matters

We’ll now update the helper function from previous video with more context. Observe how the output improves as we provide the model with more context about the dataset.


In [4]:
# System prompt that tells the model exactly what kind of code it is allowed to return.
# We restrict it to pandas code that operates on the existing DataFrame `df` and
# stores the final output in `result_df`, with no imports or extra text.

EDA_SUMMARY_PROMPT = (
    "You are an EDA assistant. "
    "Write pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. Avoid deprecated arguments or methods. "
    "Focus on descriptive statistics: shape, dtypes, missing values, "
    "unique counts, central tendency and spread. "
    "Use only valid pandas operations and function names. "
    "Do not invent custom aggregation names or shorthand labels inside pandas methods. "
    "If you need quartiles or similar statistics, compute them explicitly with valid pandas code such as quantile(). "
    "Store the final result in `result_df` as a DataFrame and not multiple variables."
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations."
)


In [5]:
QUESTION = "Identify any columns with mixed or inconsistent data formats"

#### v1 — Column names + dtypes

In [6]:
context_v1 = f"Columns: {list(df.columns)}"

In [7]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"{context_v1}\n\nQuestion: {QUESTION}",  # Makes the output deterministic
    config={
        "temperature": 0.0,
        "seed": 42,
        "system_instruction": EDA_SUMMARY_PROMPT,
    },
)


In [8]:
# Extract executable Python from the model response.
response.text

"```python\nimport pandas as pd\n\nmixed_type_columns_info = []\n\nfor col in df.columns:\n    # Get unique Python types of non-null values in the column.\n    # .dropna() removes NaN, None, and pd.NA values before checking types.\n    # .apply(type) gets the Python type object for each element.\n    # .unique() returns an array of unique type objects.\n    unique_types = df[col].dropna().apply(type).unique()\n\n    # If there's more than one unique type, it indicates mixed data formats.\n    if len(unique_types) > 1:\n        # Convert type objects to their string names for better readability and sort them.\n        type_names = sorted([t.__name__ for t in unique_types])\n        mixed_type_columns_info.append({'Column Name': col, 'Detected Types': ', '.join(type_names)})\n\n# Create the result DataFrame from the collected information.\nresult_df = pd.DataFrame(mixed_type_columns_info)\n\n# If no mixed types are found, ensure result_df is an empty DataFrame with the correct columns.\n

In [9]:
text = (response.text or "").strip()
match = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
code = match.group(1).strip() if match else text


In [10]:
code


"import pandas as pd\n\nmixed_type_columns_info = []\n\nfor col in df.columns:\n    # Get unique Python types of non-null values in the column.\n    # .dropna() removes NaN, None, and pd.NA values before checking types.\n    # .apply(type) gets the Python type object for each element.\n    # .unique() returns an array of unique type objects.\n    unique_types = df[col].dropna().apply(type).unique()\n\n    # If there's more than one unique type, it indicates mixed data formats.\n    if len(unique_types) > 1:\n        # Convert type objects to their string names for better readability and sort them.\n        type_names = sorted([t.__name__ for t in unique_types])\n        mixed_type_columns_info.append({'Column Name': col, 'Detected Types': ', '.join(type_names)})\n\n# Create the result DataFrame from the collected information.\nresult_df = pd.DataFrame(mixed_type_columns_info)\n\n# If no mixed types are found, ensure result_df is an empty DataFrame with the correct columns.\nif result_d

In [11]:
# Define a restricted execution environment.
# The generated code can only access pandas (pd) and the DataFrame (df).
env = {"pd": pd, "df": df.copy()}

# Execute the generated code in the restricted environment.
exec(code, env, env)

# The model is instructed to store the final output in `result_df`.
env.get("result_df")


,Column Name,Detected Types


Using `exec()` to run LLM-generated code is powerful but requires care. In a production environment you'd want to add error handling (try/except around the exec), validate the output type, and potentially sandbox the execution. For learning purposes, the pattern here is fine.

#### v2 — Column names + dtypes + sample rows

In [12]:
context_v2 = (
    f"Columns: {list(df.columns)}\n"
    f"Dtypes:\n{df.dtypes.to_string()}\n"
    f"Sample (10 rows):\n{df.head(10).to_string()}"
)


In [14]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"{context_v2}\n\nQuestion: {QUESTION}",
    config={
        "temperature": 0.0,
        "seed": 42,
        "system_instruction": EDA_SUMMARY_PROMPT,
    },
)


In [15]:
text = (response.text or "").strip()
match = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
code = match.group(1).strip() if match else text
env = {"pd": pd, "df": df.copy()}
exec(code, env, env)
env.get("result_df")


,Column Name,Inconsistency Type,Details
0,None,No major inconsistencies detected,All columns appear consistent in type and format based on initial checks.


Sample rows allow the model to see actual values, which helps it detect issues like mixed formats, encoded categories, or unexpected string patterns.

## 3 - The EDA Summary Helper

In [16]:
def eda_summary_helper(question, frame, show_code=False):
    """Ask a question about a DataFrame; get back a DataFrame.

    Args:
        question:      Plain-English EDA question.
        frame:         Input DataFrame. Not modified in place.
        show_code:     If True, print generated code before executing.

    Returns:
        result DataFrame.
    """
    prompt = (
        f"Columns: {list(frame.columns)}"
        f"Dtypes:{frame.dtypes.to_string()}"
        f"Sample (10 rows):{frame.head(10).to_string()}"
        f"Question: {question}"
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,  # Makes the output deterministic
            "seed": 42,
            "system_instruction": EDA_SUMMARY_PROMPT,  # System level instructions that define rules or behavior for the model
        },
    )

    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---")

    # Define a restricted execution environment.
    # The generated code can only access pandas (pd) and the DataFrame (df).
    env = {"pd": pd, "df": frame.copy()}

    # Execute the generated code in the restricted environment.
    exec(code, env, env)

    # The model is instructed to store the final output in `result_df`.
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result

In [17]:
result_df = eda_summary_helper(
    "Give me a summary of the DataFrame with descriptive statistics", df, show_code=True
)
result_df

--- Generated Code ---
summary_rows = []
total_rows = df.shape[0]

for col in df.columns:
    col_summary = {
        'Column': col,
        'Dtype': df[col].dtype,
        'Non-Null Count': df[col].count(),
        'Missing Count': df[col].isnull().sum(),
        'Missing %': (df[col].isnull().sum() / total_rows) * 100,
        'Unique Count': df[col].nunique()
    }

    if pd.api.types.is_numeric_dtype(df[col]):
        col_summary['Mean'] = df[col].mean()
        col_summary['Std'] = df[col].std()
        col_summary['Min'] = df[col].min()
        col_summary['25%'] = df[col].quantile(0.25)
        col_summary['50%'] = df[col].quantile(0.50)
        col_summary['75%'] = df[col].quantile(0.75)
        col_summary['Max'] = df[col].max()
        col_summary['Top'] = pd.NA
        col_summary['Freq'] = pd.NA
    else: # Categorical/Object type
        col_summary['Mean'] = pd.NA
        col_summary['Std'] = pd.NA
        col_summary['Min'] = pd.NA
        col_summary['25%'] = pd.NA
   

,Dtype,Non-Null Count,Missing Count,Missing %,Unique Count,Mean,Std,Min,25%,50%,75%,Max,Top,Freq
Column,,,,,,,,,,,,,,
Employee ID,int64,5030,0,0.000000,5000,3500.063419,1442.908631,1001,2251.25,3500.5,4747.75,6000,NaN,<NA>
age,int64,5030,0,0.000000,40,34.139165,8.511891,21,28.0,33.0,39.0,60,NaN,<NA>
gender,str,5030,0,0.000000,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Male,2429
department,str,5030,0,0.000000,6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Engineering,1433
department_code,str,5030,0,0.000000,6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ENG-02,1433
JobTitle,str,5030,0,0.000000,42,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Logistics Coordinator,165
job_level,int64,5030,0,0.000000,5,2.375149,1.192965,1,1.0,2.0,3.0,5,NaN,<NA>
Education,str,5030,0,0.000000,4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Bachelor's,2286
MonthlyIncome,int64,5030,0,0.000000,3680,7419.086879,2516.283477,3201,5520.5,6878.5,8691.75,20659,NaN,<NA>


## 4 - Save for Reuse



In [18]:
import inspect

components = [
    "import os",
    "import re",
    "import pandas as pd",
    "from google import genai",
    "from dotenv import load_dotenv",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    f"EDA_SUMMARY_PROMPT = {repr(EDA_SUMMARY_PROMPT)}",
    "",
    inspect.getsource(eda_summary_helper),
]

with open("eda_summary_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved eda_summary_helper.py")


Saved eda_summary_helper.py
